# 🚀 EchoVision: Decoupled Audio-Visual RAG for Video QA
### ⚡ Google Colab T4 GPU Edition (Multi-Model & Serverless API Enabled)

This notebook provides a complete pipeline to run **EchoVision** with state-of-the-art multi-modal models:
- **LLM Generators (via Serverless API / Local)**: `qwen2.5-v1-72b-instruct`, `gemma-4-31b`, `phi-3.5-vision-instruct`
- **Visual Captioning Models**: `Salesforce/blip-image-captioning-base`, `Salesforce/blip-image-captioning-large`, `HuggingFaceTB/SmolVLM-256M-Instruct`, `wraps/moondream-caption`
- **Audio Embedding Model**: `FacebookAI/roberta-base` (768-dim normalized representations)
- **One-by-One Evaluation**: Automated benchmark runner (`run_model_experiments.py`) to evaluate and compare each model sequentially.

> **GPU Setup**: Designed for Google Colab **Tesla T4 (16 GB)** using sequential memory-safe extraction and remote API inference for heavy models (31B–72B).

## 1️⃣ Hardware Verification: Confirm Tesla T4 GPU
Verify GPU allocation, VRAM capacity, and PyTorch CUDA environment.


In [ ]:
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total VRAM      : {vram_gb:.2f} GB")
    print(f"CUDA Capability : {torch.cuda.get_device_capability(0)}")
    print("[✓] Tesla T4 GPU runtime verified successfully!")
else:
    print("[!] GPU not detected. Please select 'Runtime' -> 'Change runtime type' -> 'T4 GPU'.")
print("=" * 60)


## 2️⃣ Clone Repository from GitHub
Clone the official repository from [https://github.com/G-shubham18/online_EVL.git](https://github.com/G-shubham18/online_EVL.git) and navigate to the project directory.


In [ ]:
import os

REPO_URL = "https://github.com/G-shubham18/online_EVL.git"
REPO_DIR = "online_EVL"

# Clone repository into Colab environment or pull latest updates
if not os.path.exists("main.py"):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning EchoVision from {REPO_URL} ...")
        !git clone {REPO_URL}
    %cd {REPO_DIR}

print("Pulling latest framework code from GitHub...")
!git pull

print("Current Working Directory:", os.getcwd())
!ls -lh


## 3️⃣ Install System & Python Dependencies
Install system multimedia libraries (`ffmpeg`, `libsndfile1`) and multi-modal Python packages from `requirements.txt`.


In [ ]:
# 1. System packages for video & audio processing
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1

# 2. Python requirements
!pip install -q -r requirements.txt

# Verify key multi-modal libraries
import faster_whisper
import transformers
import scenedetect
import chromadb
import faiss
import rank_bm25
import timm
import openai
print("\n[✓] All dependencies installed and verified successfully!")


## 4️⃣ Configure API Keys (Hugging Face & OpenAI)
Configure your **Hugging Face API Token** (`HF_TOKEN`) to run large models (`qwen2.5-v1-72b-instruct`, `gemma-4-31b`, `phi-3.5-vision-instruct`) and captioning models remotely via serverless inference without GPU VRAM constraints.

*Tip*: You can save `HF_TOKEN` and `OPENAI_API_KEY` in Google Colab's **🔑 Secrets** tab (left sidebar) for automatic loading.

In [ ]:
import os, getpass

# 1. Hugging Face API Token (Recommended for 72B / 31B LLMs and Remote Inference)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = os.getenv('HF_TOKEN')

if not hf_token:
    hf_token = getpass.getpass('Enter your Hugging Face Token (hf_..., press Enter to skip): ').strip()

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['USE_API'] = '1'
    print('[✓] Hugging Face Token configured! Serverless API inference enabled.')
    try:
        from huggingface_hub import HfApi
        user_info = HfApi(token=hf_token).whoami()
        print(f"[✓] Authenticated as Hugging Face user: {user_info.get('name', 'User')}")
    except Exception as e:
        print(f"[!] Token validation note: {e}")
else:
    print('[i] No Hugging Face Token provided. Falling back to local offline model execution.')

# 2. Optional OpenAI API Key (for GPT-4o-mini baseline if desired)
openai_key = None
try:
    from google.colab import userdata
    openai_key = userdata.get('OPENAI_API_KEY')
except Exception:
    pass

if not openai_key:
    openai_key = os.getenv('OPENAI_API_KEY')

if openai_key:
    os.environ['OPENAI_API_KEY'] = openai_key
    print('[✓] OpenAI API Key configured.')


## 5️⃣ Stage 3 Local LLM Setup (Ollama Daemon Fallback)
If running without an OpenAI API key, install and start Ollama in the background with `qwen2.5:1.5b` (takes only ~1.5 GB VRAM on T4).


In [ ]:
# Install Ollama binary in Colab (skip if already using GPT-4o-mini)
if os.getenv("LLM_BACKEND") != "gpt":
    !curl -fsSL https://ollama.com/install.sh | sh

    import subprocess, time, requests
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)

    # Pull lightweight Qwen2.5 model optimized for T4 GPU
    !ollama pull qwen2.5:1.5b

    try:
        res = requests.get("http://localhost:11434/api/tags", timeout=2.0)
        models = [m['name'] for m in res.json().get('models', [])]
        print(f"[✓] Ollama active! Available models: {models}")
    except Exception:
        print("[!] Ollama not running; falling back to in-process Hugging Face LLM backend.")
else:
    print("[i] GPT-4o-mini is active; skipping local Ollama setup.")


## 6️⃣ Multi-Model & Hardware Configuration
Select your active model combination from the supported suites:
- **LLM Options**: `qwen2.5-v1-72b-instruct` | `gemma-4-31b` | `phi-3.5-vision-instruct`
- **Captioning Options**: `Salesforce/blip-image-captioning-base` | `Salesforce/blip-image-captioning-large` | `HuggingFaceTB/SmolVLM-256M-Instruct` | `wraps/moondream-caption`
- **Audio Embedding Option**: `FacebookAI/roberta-base`
- **Sequential Ingestion**: `CONCURRENT_INGESTION=0` (keeps peak VRAM < 7.5 GB on T4)

In [ ]:
import os

# Active Model Selection
os.environ["LLM_MODEL"] = "qwen2.5-v1-72b-instruct"           # qwen2.5-v1-72b-instruct | gemma-4-31b | phi-3.5-vision-instruct
os.environ["CAPTION_MODEL"] = "Salesforce/blip-image-captioning-base" # blip-base | blip-large | SmolVLM | moondream
os.environ["AUDIO_EMBED_MODEL"] = "FacebookAI/roberta-base"   # FacebookAI/roberta-base (768-dim)
os.environ["USE_API"] = "1"

# T4 Memory & Feature Extraction Parameters
os.environ["WHISPER_MODEL"] = "large-v3-turbo"
os.environ["AST_MODEL"] = "MIT/ast-finetuned-audioset-10-10-0.4593"
os.environ["TEXT_EMBEDDING_MODEL"] = "BAAI/bge-large-en-v1.5"
os.environ["RTDETR_MODEL"] = "PekingU/rtdetr_r50vd"
os.environ["GROUNDING_DINO_MODEL"] = "IDEA-Research/grounding-dino-tiny"
os.environ["ENABLE_BM25"] = "1"
os.environ["BM25_WEIGHT"] = "0.4"
os.environ["CONCURRENT_INGESTION"] = "0"  # Memory-safe sequential extraction for T4

import config
print("=" * 65)
print(f"Device                 : {config.DEVICE} (GPU: {config.IS_GPU})")
print(f"Active LLM Model       : {config.ACTIVE_LLM_MODEL}")
print(f"Active Caption Model   : {config.ACTIVE_CAPTION_MODEL}")
print(f"Active Audio Embed Model: {config.ACTIVE_AUDIO_EMBED_MODEL}")
print(f"API Inference Active   : {config.USE_API} (HF_TOKEN set: {bool(config.HF_TOKEN)})")
print(f"Ingestion Mode         : {'Sequential (Memory-Safe for T4)' if not config.CONCURRENT_INGESTION else 'Concurrent'}")
print("=" * 65)


## 7️⃣ Inspect Videos & Question Dataset
Scan `smoketest/videos` and verify questions schema in `smoketest/json`.

> **Note**: Due to GitHub file-size limits, large `.mp4` video files are gitignored. If `smoketest/videos/` is empty upon cloning, this cell will automatically generate a sample test video clip (`v_96vBhCFBbQk.mp4`) matching the dataset question using FFmpeg so you can run the complete pipeline immediately!


In [ ]:
import os, json
from ingestion import discover_videos

videos_dir = "smoketest/videos"
os.makedirs(videos_dir, exist_ok=True)
videos = discover_videos(videos_dir)

if not videos:
    print("[!] No videos found in smoketest/videos (videos are gitignored in GitHub repository).")
    print("[+] Generating a 5-second test video clip with audio tone via FFmpeg...")
    sample_clip = os.path.join(videos_dir, "v_96vBhCFBbQk.mp4")
    !ffmpeg -y -f lavfi -i testsrc=duration=5:size=640x360:rate=25 -f lavfi -i sine=frequency=440:duration=5 -c:v libx264 -c:a aac "{sample_clip}" -loglevel error
    videos = discover_videos(videos_dir)
    print(f"[✓] Created test video: {sample_clip}")

print(f"\nDiscovered {len(videos)} video(s) in {videos_dir}:")
for v in videos[:6]:
    print(f"  - {os.path.basename(v)}")
if len(videos) > 6:
    print(f"  ... and {len(videos) - 6} more")

json_file = "smoketest/json/smoketest_questions.json"
if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    print(f"\nTotal Questions in Dataset: {len(q_data)}")
    print("Sample Question Entry:")
    print(json.dumps(q_data[1] if len(q_data) > 1 else q_data[0], indent=2))


## 8️⃣ Run Stage 1 (Offline Feature Extraction & Isolated Vector Indexing)
- Extracts Speech (`Whisper large-v3-turbo`) with word-level timestamps
- Extracts Audio Events (`AST` Audio Spectrogram Transformer)
- Embeds Audio Texts using `FacebookAI/roberta-base` into isolated ChromaDB vector stores
- Detects and tracks key objects (`RT-DETR` + `GroundingDINO` + `SimpleSort`)
- Captions keyframes using selected visual model (`Salesforce/blip-image-captioning-base` or API)
- Embeds visual captions with `BAAI/bge-large-en-v1.5` into isolated FAISS indices

In [ ]:
!python ingestion.py --dataset_dir smoketest --sequential \
    --caption_model "$CAPTION_MODEL" \
    --audio_embed_model "$AUDIO_EMBED_MODEL" \
    --use_api


## 9️⃣ Run Stage 2 & 3 (Decoupled Retrieval & Answer Generation)
- Estimates audio-visual modality dependency $\beta(q)$
- Decoupled parallel retrieval from ChromaDB (audio) and FAISS (visual)
- Audio-aware cross-encoder re-ranking (`bge-reranker-large`) & BM25 hybrid ranking
- Temporal NMS deduplication & sufficiency gate loopback
- Generates grounded answers with selected LLM (`qwen2.5-v1-72b-instruct` / `gemma-4-31b` / `phi-3.5-vision-instruct`)

In [ ]:
!python main.py --dataset_dir smoketest --output_dir output \
    --llm_model "$LLM_MODEL" \
    --caption_model "$CAPTION_MODEL" \
    --audio_embed_model "$AUDIO_EMBED_MODEL" \
    --use_api


## 🔟 View Predictions & Evaluation Metrics Summary
Inspect exact match and relaxed accuracy metrics across spatial, temporal, spatiotemporal, and audio categories:


In [ ]:
import os, json
import pandas as pd

eval_path = "output/evaluation_summary.json"
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    print("=" * 60)
    print("               EVALUATION METRICS REPORT               ")
    print("=" * 60)
    metrics = eval_data.get("metrics", {})
    print(f"Total Evaluated Questions   : {metrics.get('total_evaluated', 0)}")
    print(f"Exact Match Accuracy        : {metrics.get('exact_match_accuracy', 0)}%")
    print(f"Relaxed Substring Accuracy  : {metrics.get('relaxed_accuracy', 0)}%")
    if metrics.get("category_breakdown"):
        print("\nCategory Breakdown:")
        for cat, c_res in metrics["category_breakdown"].items():
            print(f"  - {cat:<20}: {c_res['accuracy']:>6.2f}% ({c_res['correct']}/{c_res['total']})")
    print("=" * 60)

pred_path = "output/smoketest_questions.json"
if os.path.exists(pred_path):
    with open(pred_path, "r", encoding="utf-8") as f:
        preds = json.load(f)
    df = pd.DataFrame(preds)
    cols = [c for c in ["video_id", "category", "question", "predicted_answer", "ground_truth_answer"] if c in df.columns]
    print(f"\nSample Predictions Table ({len(df)} total):")
    display(df[cols].head(15))


## 1️⃣1️⃣ Run Model Experiments One by One (`run_model_experiments.py`)
Benchmark each model **one by one** with isolated outputs and automatic evaluation reporting:
- **LLMs one by one**: `qwen2.5-v1-72b-instruct` ➔ `gemma-4-31b` ➔ `phi-3.5-vision-instruct`
- **Caption models one by one**: `blip-base` ➔ `blip-large` ➔ `SmolVLM` ➔ `moondream`
- **Audio embed model**: `FacebookAI/roberta-base`
- Outputs are isolated into `output/model_benchmarks/<model_name>/` and summarized in `master_benchmark_summary.md`.

In [ ]:
# Run all LLMs one by one via API
!python run_model_experiments.py --mode llm --use_api --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest

# To test captioning models one by one, uncomment:
# !python run_model_experiments.py --mode caption --use_api --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest

# To test audio embedding model, uncomment:
# !python run_model_experiments.py --mode audio --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest

# To run all models sequentially in one run, uncomment:
# !python run_model_experiments.py --mode all --use_api --videos_dir smoketest/videos --json_dir smoketest/json --auto_ingest


In [ ]:
# Display Master Benchmark Markdown Report
import os
from IPython.display import Markdown, display

summary_path = "output/model_benchmarks/master_benchmark_summary.md"
if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print("Benchmark summary not found yet. Run an experiment above to generate results.")


## 1️⃣2️⃣ Run Scientific Ablation Benchmark Suite (`run_ablations.py`)
Run all 4 architectural ablations + baseline in one automated sweep:
1. `joint_store` (Unified vs Decoupled vector stores)
2. `no_audio_lift` (Re-ranking without audio lift)
3. `fixed_cutoff` (Fixed depth vs Adaptive sufficiency gate)
4. `no_audio` (Audio removed entirely - visual-only baseline)


In [ ]:
!python run_ablations.py --dataset_dir smoketest --output_dir output/ablations


## 1️⃣3️⃣ Interactive Demo: Query Any Video with Custom Questions
Test any video interactively! Specify a video path and question below:


In [ ]:
import os
from ingestion import discover_videos, Stage1Ingestor
from main import answer_question_for_video
from stage2_online.question_classifier import QuestionClassifier
from stage2_online.deduplicator import Deduplicator
from stage2_online.reranker import ReRanker
from stage2_online.sufficiency_gate import SufficiencyGate
from stage3_generator.generator import Generator

sample_videos = discover_videos("smoketest/videos")
if sample_videos:
    test_video = sample_videos[0]
    test_question = "what is behind the person in blue?"
    
    llm_choice = os.getenv("LLM_MODEL", "qwen2.5-v1-72b-instruct")
    caption_choice = os.getenv("CAPTION_MODEL", "Salesforce/blip-image-captioning-base")
    audio_choice = os.getenv("AUDIO_EMBED_MODEL", "FacebookAI/roberta-base")
    
    print(f"Selected Video: {test_video}")
    print(f"Question      : {test_question}")
    print(f"Active Models : LLM={llm_choice} | Caption={caption_choice} | Audio={audio_choice}\n")
    
    # 1. Run Stage 1 (instant if already cached)
    ingestor = Stage1Ingestor(
        caption_model=caption_choice,
        audio_embed_model=audio_choice,
        use_api=os.getenv("USE_API", "1").lower() in ("1", "true", "yes"),
        hf_token=os.getenv("HF_TOKEN")
    )
    indexer, reused, err = ingestor.process_single_video(test_video)
    
    if indexer:
        # 2. Shared online reasoning components
        shared_components = {
            'qc': QuestionClassifier(),
            'dedup': Deduplicator(),
            'reranker': ReRanker(),
            'gate': SufficiencyGate(),
            'generator': Generator(
                model=llm_choice,
                use_api=os.getenv("USE_API", "1").lower() in ("1", "true", "yes"),
                hf_token=os.getenv("HF_TOKEN")
            )
        }
        
        # 3. Answer question
        answer = answer_question_for_video(indexer, test_question, shared_components)
        print("\n" + "=" * 50)
        print(f"PREDICTED ANSWER: {answer}")
        print("=" * 50)


## 1️⃣4️⃣ Export & Download Output Files
Package evaluation summary, prediction JSONs, and ablation reports into a single zip and download directly.


In [ ]:
import os, shutil
from google.colab import files

if os.path.exists("output"):
    shutil.make_archive("echovision_colab_results", 'zip', "output")
    files.download("echovision_colab_results.zip")
    print("[✓] Results archive (echovision_colab_results.zip) downloaded successfully!")
else:
    print("[!] Output directory 'output/' does not exist yet. Please run Stage 2 & 3 first.")
